# Import software libraries

In [66]:
# Import required libraries.
import sys           # Read system parameters.
import pandas as pd  # Manipulate and analyze data.
import sqlite3       # Manage SQL databases.

# Summarize software libraries used.
print('Libraries used in this project:')
print('- Python {}'.format(sys.version))
print('- pandas {}'.format(pd.__version__))
print('- sqlite3 {}'.format(sqlite3.sqlite_version))

Libraries used in this project:
- Python 3.10.9 | packaged by Anaconda, Inc. | (main, Mar  1 2023, 18:18:15) [MSC v.1916 64 bit (AMD64)]
- pandas 1.5.3
- sqlite3 3.41.2


# Examine the database

In [67]:
# Connect to SQLite database.


#conn = sqlite3.connect('/home/jovyan/work/ETL/data/prod_sample.db')
#data in my local computeur
conn = sqlite3.connect('prod_sample.db')

In [68]:
# List all the tables in the database.
cursor = conn.cursor()

In [69]:
# Exécution de la requête pour obtenir la liste des tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

# Récupération des résultats
tables = cursor.fetchall()

# Affichage des noms des tables
for table in tables:
    print(table[0])

stock_description
online_retail_history


# Read data from the `online_retail_history` table

In [70]:
# Write the query to be executed that selects everything from the online_retail_history table.

# Exécution de la requête pour sélectionner tout depuis la table online_retail_history
cursor.execute("SELECT * FROM online_retail_history;")

# Récupération des résultats
#rows = cursor.fetchall()

# Affichage des résultats
#for row in rows:
#    print(row)


In [71]:
# Use the read_sql function in pandas to read a query into a DataFrame.

# Lecture de la requête SQL dans un DataFrame
df = pd.read_sql("SELECT * FROM online_retail_history;", conn)

In [72]:
# Preview the first five rows of the data.
df.head(5)

,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08
2,536373,85123A,6,2010-12-01 09:02:00,2.55,u1785,United Kingdom,15.30
3,536375,85123A,6,2010-12-01 09:32:00,2.55,u1785,United Kingdom,15.30
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50


In [73]:
# Get the shape of the data.
df.shape

(15321, 8)

# Read data from the `stock_description` table

In [74]:
# Write the query to be executed that selects everything from the online_retail_history table.
# Write the query to be executed that selects everything from the online_retail_history table.

# Exécution de la requête pour sélectionner tout depuis la table online_retail_history
cursor.execute("SELECT * FROM stock_description;")

# Récupération des résultats
#rows = cursor.fetchall()

# Affichage des résultats
#for row in rows:
#    print(row)

In [75]:
# Use the read_sql function in pandas to read a query into a DataFrame.
# Lecture de la requête SQL dans un DataFrame
df_des = pd.read_sql("SELECT * FROM stock_description;", conn)

In [76]:
# Preview the first five rows of the data.
df_des.head(5)

,StockCode,Description
0,10002,INFLATABLE POLITICAL GLOBE
1,10080,GROOVY CACTUS INFLATABLE
2,10120,DOGGY RUBBER
3,10123C,HEARTS WRAPPING TAPE
4,10124A,SPOTS ON RED BOOKCOVER TAPE


In [77]:
# Get the shape of the data.

df_des.shape

(3952, 2)

# Aggregate the `online_retail_history` and `stock_description` datasets

In [78]:
# Write a query to aggregate the two datasets so that you have the stock descriptions as well as the stock code.

query = """ SELECT * FROM online_retail_history JOIN stock_description ON (online_retail_history.StockCode = stock_description.StockCode); """

#data = cursor.execute(query)
# Récupération des résultats
#rows = data.fetchall()

# Affichage des résultats
#for row in rows:
#    print(row)

In [79]:
# Use the read_sql function in pandas to read a query into a DataFrame.
df_agre = pd.read_sql(query, conn)

In [80]:
# Preview the first five rows of the data.
df_agre.head(5)

,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount,StockCode,Description
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08,84879,ASSORTED COLOUR BIRD ORNAMENT
2,536373,85123A,6,2010-12-01 09:02:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
3,536375,85123A,6,2010-12-01 09:32:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50,20725,LUNCH BAG RED RETROSPOT


In [83]:
# Get the shape of the data.

df_agre.shape

(17032, 10)

# Identify and fix corrupt or unusable data

In [85]:
# Check the value counts of the "Description" field.

description_counts = df_agre['Description'].value_counts()
description_counts

CREAM HANGING HEART T-LIGHT HOLDER    2174
JUMBO BAG RED RETROSPOT               1960
?                                     1711
REGENCY CAKESTAND 3 TIER              1711
PARTY BUNTING                         1615
LUNCH BAG RED RETROSPOT               1421
ASSORTED COLOUR BIRD ORNAMENT         1405
POPCORN HOLDER                        1329
LUNCH BAG  BLACK SKULL.               1271
SET OF 3 CAKE TINS PANTRY DESIGN      1257
PACK OF 72 RETROSPOT CAKE CASES       1178
Name: Description, dtype: int64

In [88]:
# Remove rows where "Description" is just a question mark (?).

df_filtered = df_agre[df_agre['Description'] != '?']





# Preview the first five rows of the data.
df_filtered.head(5)


,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount,StockCode,Description
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08,84879,ASSORTED COLOUR BIRD ORNAMENT
2,536373,85123A,6,2010-12-01 09:02:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
3,536375,85123A,6,2010-12-01 09:32:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50,20725,LUNCH BAG RED RETROSPOT


# Identify and remove duplicates

In [89]:
# Identify all duplicated data.

duplicates = df_filtered[df_filtered.duplicated(['Description'], keep=False)]



In [90]:
# Print the duplicated data.
duplicates


,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount,StockCode,Description
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08,84879,ASSORTED COLOUR BIRD ORNAMENT
2,536373,85123A,6,2010-12-01 09:02:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
3,536375,85123A,6,2010-12-01 09:32:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50,20725,LUNCH BAG RED RETROSPOT
...,...,...,...,...,...,...,...,...,...,...
17027,581538,20727,1,2011-12-09 11:34:00,1.65,u14446,United Kingdom,1.65,20727,LUNCH BAG BLACK SKULL.
17028,581538,20725,1,2011-12-09 11:34:00,1.65,u14446,United Kingdom,1.65,20725,LUNCH BAG RED RETROSPOT
17029,581579,22197,24,2011-12-09 12:19:00,0.85,u17581,United Kingdom,20.40,22197,POPCORN HOLDER
17030,581583,20725,40,2011-12-09 12:23:00,1.45,u13777,United Kingdom,58.00,20725,LUNCH BAG RED RETROSPOT


In [95]:
# Remove the duplicated data.

df_unique = df_filtered.drop_duplicates(subset=['Description'], keep='first')




# Preview the first five rows of the data.
df_unique.head(5)


,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount,StockCode,Description
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08,84879,ASSORTED COLOUR BIRD ORNAMENT
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50,20725,LUNCH BAG RED RETROSPOT
5,536378,21212,120,2010-12-01 09:37:00,0.42,u14688,United Kingdom,50.40,21212,PACK OF 72 RETROSPOT CAKE CASES
6,536386,85099B,100,2010-12-01 09:57:00,1.65,u16029,United Kingdom,165.00,85099B,JUMBO BAG RED RETROSPOT


# Correct date formats

In [96]:
# Get the data types for every column in the DataFrame.

data_types = df_unique.dtypes

In [98]:
# Convert "InvoiceDate" to a "%Y-%m-%d" datetime format.
df_unique['InvoiceDate'] = \
pd.to_datetime(df_unique['InvoiceDate'], format='%Y-%m-%d', errors='coerce')


df_unique

C:\Users\HP\AppData\Local\Temp\ipykernel_5424\2602136230.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique['InvoiceDate'] = \


,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount,StockCode,Description
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08,84879,ASSORTED COLOUR BIRD ORNAMENT
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50,20725,LUNCH BAG RED RETROSPOT
5,536378,21212,120,2010-12-01 09:37:00,0.42,u14688,United Kingdom,50.40,21212,PACK OF 72 RETROSPOT CAKE CASES
6,536386,85099B,100,2010-12-01 09:57:00,1.65,u16029,United Kingdom,165.00,85099B,JUMBO BAG RED RETROSPOT
8,536390,22197,100,2010-12-01 10:19:00,0.72,u17511,United Kingdom,72.00,22197,POPCORN HOLDER
19,536404,20727,10,2010-12-01 11:29:00,1.65,u16218,United Kingdom,16.50,20727,LUNCH BAG BLACK SKULL.
37,536477,22423,16,2010-12-01 12:27:00,10.95,u1621,United Kingdom,175.20,22423,REGENCY CAKESTAND 3 TIER
181,536864,47566,1,2010-12-03 11:27:00,9.32,None,United Kingdom,9.32,47566,PARTY BUNTING
778,538654,22720,6,2010-12-13 15:13:00,4.95,u13089,United Kingdom,29.70,22720,SET OF 3 CAKE TINS PANTRY DESIGN


In [99]:
# Get the data types for every column in the converted DataFrame.

data_types = df_unique.dtypes

# Examine the table before finishing

In [100]:
# Preview the first five rows of the data.

df_unique.head(5)

,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount,StockCode,Description
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30,85123A,CREAM HANGING HEART T-LIGHT HOLDER
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08,84879,ASSORTED COLOUR BIRD ORNAMENT
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50,20725,LUNCH BAG RED RETROSPOT
5,536378,21212,120,2010-12-01 09:37:00,0.42,u14688,United Kingdom,50.40,21212,PACK OF 72 RETROSPOT CAKE CASES
6,536386,85099B,100,2010-12-01 09:57:00,1.65,u16029,United Kingdom,165.00,85099B,JUMBO BAG RED RETROSPOT


# Load the dataset into a pickle file

In [103]:
# Save the dataset as a pickle file named online_history_cleaned.pickle.

df_final = df_unique.to_pickle('prod_sample.pickle')


In [104]:
df_final

In [ ]:
# Close any connections to the database.
conn.close()
